## Design Pattern: Typeclass vs Methods

- In Scala, since we commonly work with generic types, we often need to have a single function name that is capable of changing behaviour depending on the type we are working with

- Let's suppose we have this set up:

```java
sealed trait Shape
case Circle(radius: Double) extends Shape
case Square(side: Double) extends Shape
```

- We want a function `areaOf` that works for any shape

- A Square has an area, and a Circle has an area, but they compute it differently

### Usual approach: Typeclass

- Capture "how to compute the area of an `S`" in its own trait, and give one instance per type:

```java
	trait HasArea[S] {
		def area(s: S): Double
	}

	given HasArea[Circle] with {
		def area(c: Circle): Double = math.Pi * c.radius * c.radius
	}
																						
	given HasArea[Square] with {
		def area(q: Square): Double = q.side * q.side
	}

	def areaOf[S](s: S)(using a: HasArea[S]): Double = a.area(s)
```      

- At each call site, the compiler looks at the argument's type, finds the matching `given`, and passes it in as `a`. This happens at **compile time*

```java
	val circle5 = Circle(5)
	val square5 = Square(5)

	areaOf(circle5) // compiler sees Circle, finds HasArea[Circle]
	areaOf(square5) // compiler sees Square, finds HasArea[Square]
```

### Complication 1: the type is only known at runtime

- This works because the code itself says `Circle(5)` and `Square(5)`, so the compiler knows each type

- Suppose instead the shape comes from JSON:

```java                                                                             
	val s: Shape = fromJsonString[Shape]("""{"type": "Circle", "radius": 2}""")          
	areaOf(s)   // compile error: No given instance of type HasArea[Shape]               
```

- At runtime `s` really is a `Circle`, but the compiler only knows `s: Shape`. Which case it is gets decided when the JSON is decoded, after compilation

- The compiler has `HasArea[Circle]` and `HasArea[Square]`, but no `HasArea[Shape]`, so it can't pick one

#### Lousy Workaround: narrow with `match`

- Inside each `case`, the compiler knows the narrower type, so the `given` can be found:
  ```java
	def shapeArea(s: Shape): Double = s match {
		case c: Circle => summon[HasArea[Circle]].area(c)
		case q: Square => summon[HasArea[Square]].area(q)
	}
  ```
- This works, but:                                                                   
	- Every function that starts from a plain `Shape` needs this same `match`
	- Adding a shape means editing every one of those `match`es. Because `Shape` is sealed, the compiler warns about each one you miss, but you still have to write them all

#### Better Workaround: Adding a method on the Shape trait

- Declare `area` on `Shape` and implement it in each case class:                     
  ```java
	sealed trait Shape { def area: Double }
	case class Circle(radius: Double) extends Shape { def area = math.Pi * radius * radius }
	case class Square(side: Double) extends Shape { def area = side * side }
	val s: Shape = fromJsonString[Shape](json)
	s.area // runs Circle's or Square's area, whichever s actually is                  
  ```                                                                                  
  
- The compiler only checks that every `Shape` has an `area`. Which implementation runs is decided at **runtime**, by the object itself (virtual dispatch)
                                                                                       
- No `match`, and adding a shape means writing its `area` in one place

### Complication 2: We have methods that only some shapes support

- Let's suppose we now want to add some features to only a subset of `Shape`s

- Example: Circles can roll, squares can't. How would we represent this?

- Solution: Put the method on a sealed sub-trait that only some cases extend!

```java
	sealed trait Shape { def area: Double }
	sealed trait Rollable extends Shape { def rollDistancePerTurn: Double }

	case class Circle(radius: Double) extends Rollable {
		def area = math.Pi * radius * radius
		def rollDistancePerTurn = 2 * math.Pi * radius
	}
	case class Square(side: Double) extends Shape { def area = side * side }

	case class Wheel(shape: Rollable)
```                                 

- Inside `Wheel`, `shape.rollDistancePerTurn` always works and needs no `match`

- Decoding `{"shape": {"type": "Square", "side": 1}}` as a `Wheel` fails, because `Square` isn't `Rollable`. A bad config is rejected when it's read, not later when it's used

- The sub-trait must be `sealed`: a JSON codec (e.g. jsoniter) needs the full list of cases at compile time, and `sealed` keeps every direct subtype in the same file

- A case can extend several sub-traits if it supports several capabilities